# Module 6 Project: Google Street View — Landmark Recognition

**Use Case:** Improve navigation and user experience on Google Street View by automatically identifying and annotating landmarks.

**Description:** Develop image recognition models to recognize and label landmarks captured on Google Street View. This project aims to enhance navigation accuracy and usability, providing users with enriched geographic information and improving overall user satisfaction with Google's mapping services.

**Dataset:** [Google Landmark Recognition 2021 (Kaggle)](https://www.kaggle.com/c/landmark-recognition-2021/data?select=train.csv)

**Pipeline:** Load `train.csv` → undersample/oversample to balance classes → build a CNN from scratch (5 conv blocks + dense head) with built-in data augmentation → train with EarlyStopping / ReduceLROnPlateau / ModelCheckpoint → evaluate → visualize predictions against example images of the predicted class.

In [ ]:
# https://www.kaggle.com/c/landmark-recognition-2021/data?select=train.csv

import numpy as np
import pandas as pd
import os

import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras import layers
from tensorflow.keras.layers.experimental import preprocessing
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras import regularizers

import matplotlib.pyplot as plt

import cv2

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

import sys
import gc
import time
import seaborn as sns

SEED = 42

In [ ]:
def load_trainf():
    traindf = pd.read_csv('../input/landmark-recognition-2021/train.csv')
    traindf['img_path'] = (traindf['id'].apply(lambda r: os.path.join(
        '../input/landmark-recognition-2021/train', r[0], r[1], r[2], r + '.jpg')))

    traindf['landmark_id'] = traindf['landmark_id'].apply(lambda x: np.int32(x))

    return traindf

# Reading and resizing images from their path
def img_read_resize(img_path):
    img = plt.imread(img_path)
    img_redim = cv2.resize(img,(IMG_SIZE,IMG_SIZE))
    return img_redim

def get_size(obj, seen=None):
    """Recursively finds size of objects"""
    size = sys.getsizeof(obj)
    if seen is None:
        seen = set()
    obj_id = id(obj)
    if obj_id in seen:
        return 0
    # Important mark as seen *before* entering recursion to gracefully handle
    # self-referential objects
    seen.add(obj_id)
    if isinstance(obj, dict):
        size += sum([get_size(v, seen) for v in obj.values()])
        size += sum([get_size(k, seen) for k in obj.keys()])
    elif hasattr(obj, "__dict__"):
        size += get_size(obj.__dict__, seen)
    elif hasattr(obj, "__iter__") and not isinstance(obj, (str, bytes, bytearray)):
        size += sum([get_size(i, seen) for i in obj])
    return size

In [ ]:
traindf = load_trainf()
landmark_unique = len(traindf['landmark_id'].unique())
traindf

In [ ]:
get_size(traindf)

In [ ]:
traindf.info(memory_usage='deep')

In [ ]:
print('Data from the original training dataset \n')
print('Number of images in the dataset: ', traindf.shape[0])
print('Number of different classes: ', landmark_unique)
print('Min class: ', min(traindf['landmark_id']))
print('Max class: ', max(traindf['landmark_id']))
print('\nRepetitions of elements by class:')
print(traindf['landmark_id'].value_counts())

### Peek at some raw images (before resizing)

In [ ]:
plt.figure(figsize=(25,10))

for i in range(20):
    j = np.random.randint(0, traindf.shape[0])
    img = plt.imread((traindf['img_path'][j]))
    plt.subplot(4 , 5, i+1)
    plt.xticks([])
    plt.yticks([])
    plt.xlabel(str(img.shape[0])+'x'+str(img.shape[1]), fontweight = "bold", fontsize=16)
    plt.imshow(img)

plt.show()

### Undersampling

The full dataset has 81,313 classes, many with only 1-2 images — far too imbalanced and too large to train on directly. We restrict to a slice of `landmark_id`s and cap each class at `UNDERSAMPLING_THRESHOLD` images.

In [ ]:
IMG_SIZE = 128

MIN_CLASS = 0
MAX_CLASS = 500

traindf_s = traindf[(MIN_CLASS < traindf['landmark_id']) & (traindf['landmark_id'] <= MAX_CLASS)]

UNDERSAMPLING_THRESHOLD = 80

traindf_s = (traindf_s.groupby('landmark_id', group_keys=False).
             apply(lambda x: x.sample(n = min(len(x), UNDERSAMPLING_THRESHOLD), random_state= SEED)))
traindf_s.reset_index(inplace=True, drop=True)
traindf_s['landmark_id'].value_counts()

N_DATA = len(traindf_s['landmark_id'])
clases = traindf_s['landmark_id'].unique()
N_CLASS = len(clases)

print('Sample data number, N_DATA: '+str(N_DATA))
print('\nNumber of classes in the sample, N_CLASS: '+str(N_CLASS))
print('\nMIN class of the sample: '+str(min(traindf_s['landmark_id'])))
print('\nMAX class of the sample: '+str(max(traindf_s['landmark_id'])))
print('\nRepetitions of elements by class in the sample after Undersampling:')
print(traindf_s['landmark_id'].value_counts())

In [ ]:
traindf_s['landmark_id'].value_counts().describe()

In [ ]:
plt.figure(figsize=(25,10))

for i in range(20):
    j = np.random.randint(0, N_DATA)
    img = img_read_resize(traindf_s['img_path'][j])
    plt.subplot(4 , 5, i+1)
    plt.xticks([])
    plt.yticks([])
    plt.xlabel('landmark_id: '+ str(traindf_s['landmark_id'][j]), fontweight ="bold", fontsize=16)
    plt.imshow(img)

plt.show()

### Load and resize every sampled image into memory

In [ ]:
X = []   #Images
y = []   #Classes

for i in range(traindf_s.shape[0]):
    X.append(img_read_resize(traindf_s['img_path'][i]))
    y.append(np.array(traindf_s['landmark_id'][i]))

print('Variable types: \n')
print('X: ', type(X))
print('X elements: ', type(X[0]))
print('\ny: ', type(y))
print('y elements: ', type(y[0]))

### Label encoding, normalization, and train / validation / test split

In [ ]:
#Label Encoding

LE = LabelEncoder()
LE.fit(clases)
y_LE = LE.transform(y)

X = [n/255 for n in X]

X = np.array(X)
y_LE = np.array(y_LE)
y = np.array(y)

#Separating data for training, testing and validating
X_train, X_val, y_train, y_val = train_test_split(X, y_LE, test_size = 0.10, random_state=SEED, shuffle=True)

X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size = 0.05, random_state=SEED, shuffle=True)

training_data = len(X_train)

print('X-Images: ',len(X), 'and y labels: ', len(y_LE))
print('Training. Images in X_train: ',len(X_train), 'and labels in y_train: ',len(y_train))
print('Validation. Images in X_val: ',len(X_val), 'and labels in y_val: ',len(y_val))
print('Testing. Images in X_test: ',len(X_test), 'and labels in y_test: ',len(y_test))

### Preview the data-augmentation layers on one training image

In [ ]:
data_augmentation = tf.keras.Sequential([
    #preprocessing.RandomFlip('horizontal'),
    preprocessing.RandomZoom(height_factor=(-0.2, 0.2)),
    preprocessing.RandomContrast(0.4),
    preprocessing.RandomTranslation(height_factor= (-0.2, 0.2), width_factor=(-0.2, 0.2)),
    preprocessing.RandomRotation(factor= (-0.1, 0.1)),
    #preprocessing.RandomCrop(120, 120, seed=SEED)
])

plt.figure(figsize=(25,7))
j = 1 #np.random.randint(0, N_DATA)
img_orig = X[j]
plt.subplot(2 , 6, 1)
plt.xticks([])
plt.yticks([])
plt.xlabel('ORIGINAL. landmark_id: '+ str(y[j]), fontweight ="bold", fontsize=12)
plt.imshow(img_orig)

for i in range(11):
    img_mod = data_augmentation(img_orig)
    plt.subplot(2 , 6, i+2)
    plt.xticks([])
    plt.yticks([])
    plt.xlabel('Modification '+str(i+1), fontweight ="bold", fontsize=12)
    plt.imshow(img_mod)

plt.show()

### Oversampling

Even after undersampling, some classes have very few images. Duplicate random images from those under-represented classes so every class has at least `OVERSAMPLING_THRESHOLD` samples in the training set (this only touches `X_train` / `y_train`, never validation or test).

In [ ]:
start_time = time.time()

OVERSAMPLING_THRESHOLD = 0.2 * UNDERSAMPLING_THRESHOLD
i=0; n=0; k=0

for i in range(N_CLASS):
    cont=0
    for n in range(len(y_train)):
        if y_train[n] == i:
            cont = cont+1

    if cont < OVERSAMPLING_THRESHOLD:
        img_class_i = X_train[y_train==i]
        for k in range(int(OVERSAMPLING_THRESHOLD-cont)):
            X_train = np.concatenate((X_train,np.array([(img_class_i[np.random.randint(len(img_class_i))])])),axis = 0)
            y_train = np.append(y_train, i)

print('\nOversampling duration: %s seconds' % (time.time() - start_time))
n_data_train_def = len(X_train)
print('\nTraining data after oversampling: '+str(n_data_train_def)+' data')
print('\nThe training data has gone from: '+str(training_data)+' to:'+ str(n_data_train_def))

### CNN hyperparameters and model definition

In [ ]:
# CNN Hyperparameters
kernelSize = (3,3)        #Convolution template size
paddingType = 'same'
activationF = 'relu'      # Activation Function
poolSize = (2,2)          # Maximum pooling template size
stridesSize = (2,2)       # Template offset during maximum pooling
dropoutRate = 0.5         # Percentage of neurons that are deactivated with the Dropout layer
batchSize = 256           # Amount of data with which it is trained in each epoch
epochsSize = 1000         # Number of epochs to train
lr = 0.01                 # Learning Rate
weightDecay = regularizers.L2(0.01) # Regularization of weights

In [ ]:
def create_model():

    model = keras.Sequential([

        #DATA AUGMENTATION
        preprocessing.RandomZoom(height_factor=(-0.2, 0.2)),
        preprocessing.RandomContrast(0.4),
        preprocessing.RandomTranslation(height_factor= (-0.2, 0.2), width_factor=(-0.2, 0.2)),
        preprocessing.RandomRotation(factor= (-0.1, 0.1)),

        # MODEL BASIS ---- Feature Extraction
        # convolutional block 1
        layers.Conv2D(filters=32, kernel_size=kernelSize, strides=1, padding=paddingType, activation=activationF, input_shape=[IMG_SIZE, IMG_SIZE, 3]),
        layers.BatchNormalization(),
        layers.MaxPool2D(pool_size=poolSize, strides=stridesSize, padding=paddingType),
        # convolutional block 2
        layers.Conv2D(filters=64, kernel_size=kernelSize, strides=1, padding=paddingType, activation=activationF),
        layers.BatchNormalization(),
        layers.MaxPool2D(pool_size=poolSize, strides=stridesSize, padding=paddingType),
        # convolutional block 3
        layers.Conv2D(filters=128, kernel_size=kernelSize, strides=1, padding=paddingType, activation=activationF),
        layers.BatchNormalization(),
        layers.MaxPool2D(pool_size=poolSize, strides=stridesSize, padding=paddingType),
        # convolutional block 4
        layers.Conv2D(filters=256, kernel_size=kernelSize, strides=1, padding=paddingType, activation=activationF),
        layers.BatchNormalization(),
        layers.MaxPool2D(pool_size=poolSize, strides=stridesSize, padding=paddingType),
        # convolutional block 5
        layers.Conv2D(filters=512, kernel_size=kernelSize, strides=1, padding=paddingType, activation=activationF),
        layers.BatchNormalization(),
        layers.MaxPool2D(pool_size=poolSize, strides=stridesSize, padding=paddingType),

        # MODEL HEAD ---- Classification
        layers.Flatten(),
        layers.Dense(units = 512, activation=activationF, kernel_regularizer=weightDecay),
        layers.Dropout(rate=dropoutRate,seed=SEED),
        layers.BatchNormalization(),
        #Output Layer
        layers.Dense(units = N_CLASS, activation='softmax'),
    ])

    # OPTIMIZERS
    # The lower lr, the less sudden changes there are in metrics such as accuracy
    opt1 = tf.keras.optimizers.RMSprop(learning_rate=lr, rho=0.9, momentum=0.5, epsilon=1e-07)
    opt2 = tf.keras.optimizers.Adam(learning_rate=lr, beta_1=0.9, beta_2=0.999, epsilon=1e-07)
    opt3 = tf.keras.optimizers.SGD(learning_rate=lr, momentum=0.9, nesterov=False)            #momentum--> acelera gradiente y amortigua oscilaciones

    # Model compilation
    model.compile(
        optimizer = opt3,
        loss = 'sparse_categorical_crossentropy',
        metrics = ['sparse_categorical_accuracy']
    )

    # Callback 1
    early_stopping = EarlyStopping(
        monitor = "val_loss",
        mode = "auto",
        min_delta = 0.0001,                # minimium amount of change to count as an improvement
        patience = 150,                     # how many epochs to wait before stopping
        restore_best_weights = True)
    # Callback 2
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss', factor=0.7,
        patience=40, cooldown=1, min_lr=0.0005,
        min_delta=0.001, verbose=1)
    # Callback 3
    checkpoint = ModelCheckpoint(
        'best-weights.h5', monitor='val_loss',
        save_best_only=True, save_weights_only=True)

    # Model Training
    history = model.fit(
        X_train, y_train,
        validation_data = (X_val, y_val),      # THE LABELS ARE WITH LABEL ENCODING
        class_weight = train_classWeights,
        shuffle = True,                         # Only affects training data (1 time at first)
        batch_size = batchSize,
        steps_per_epoch = len(X_train)//batchSize,
        epochs = epochsSize,
        callbacks = [early_stopping, reduce_lr, checkpoint],
        verbose=1,
        # use_multiprocessing = True,
        # max_queue_size = 15,        # Default = 10
        # workers = 32,
    )

    return model, history

### Class weights (to further compensate for imbalance during training)

In [ ]:
from sklearn.utils import class_weight
classWeights = class_weight.compute_class_weight(class_weight ='balanced',classes = np.unique(y_train), y = y_train)
train_classWeights = dict(enumerate(classWeights))

# Calculation of weights of each class of data for sample training
_, freq = np.unique(y_train, return_counts=True)
max_freq = np.max(freq)
print ('Highest frequency: '+str(np.max(freq)))

### Train the model

In [ ]:
start_time = time.time()

model, history = create_model()

history_df = pd.DataFrame(history.history)

print('\nbatchSize = '+str(batchSize))

plt.figure(figsize=(15,5))        #Width and Height of the graphs, respectively

plt.subplot(1,2,1)
#history_df.loc[0:, ['loss', 'val_loss']].plot()
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.legend()
plt.title('Loss and Validation Loss')
plt.xlabel('batchSize = '+str(batchSize))
print(("Minimum Validation Loss: {:0.4f} in epoch {:0.0f} ").format(history_df['val_loss'].min(), history_df['val_loss'].idxmin()))

plt.subplot(1,2,2)
#history_df.loc[0:, ['accuracy', 'val_accuracy']].plot()
plt.plot(history.history['sparse_categorical_accuracy'], label='Training')
plt.plot(history.history['val_sparse_categorical_accuracy'], label='Validation')
plt.legend()
plt.title('Accuracy and Validation Accuracy')
plt.xlabel('batchSize = '+str(batchSize))
print(("Maximum Validation Accuracy: {:0.4f} in epoch {:0.0f} ").format(history_df['val_sparse_categorical_accuracy'].max(), history_df['val_sparse_categorical_accuracy'].idxmax()))

print("\nEvaluation of the model with training data")
score = model.evaluate(X_train, y_train)
print("Test loss, Test accuracy:", score[0], score[1])

print("\nModel evaluation with validation data")
score = model.evaluate(X_val, y_val)
print("Test loss, Test accuracy:", score[0], score[1])

plt.show()

print('\ntraining duration: %s minutes' % ((time.time() - start_time)/60))

In [ ]:
model.summary()

### Look at a batch of held-out test images before predicting

In [ ]:
plt.figure(figsize=(25,7))

for i in range(12):
    img = X_test[i]  #Ya se hizo el shuffle con el split
    plt.subplot(2 , 6, i+1)
    plt.xticks([])
    plt.yticks([])
    plt.xlabel('Class: determined')
    plt.imshow(img)

plt.show()

### Predict on the test set

In [ ]:
predict = model.predict(X_test, verbose=1)

In [ ]:
y_pred = []
confidence = []

for i in range(len(predict)):
    y_pred.append(np.argmax(predict[i]))
    confidence.append(predict[i][y_pred[i]].round(2))

y_pred = LE.inverse_transform(y_pred)

In [ ]:
print("\nEvaluation of the model on test data")
score = model.evaluate(X_test, y_test)
print("Test loss, Test accuracy:", score[0], score[1])

In [ ]:
y_test = LE.inverse_transform(y_test)

### Visualize predictions

For each test image, show the predicted class / confidence / actual class alongside a few example training images that belong to the predicted class, so you can visually judge whether the prediction looks reasonable.

In [ ]:
def plot_prediction_examples(k, col=5):
    img_to_predict = X_test[k]

    # Example images belonging to the predicted class, drawn from the sampled training dataframe
    img_class_df = traindf_s[traindf_s['landmark_id'] == y_pred[k]].reset_index(drop=True)

    plt.figure(figsize=(25, 5))

    plt.subplot(1, col, 1)
    plt.xticks([])
    plt.yticks([])
    plt.title('Image to predict #'+str(k))
    plt.xlabel('Predicted class: '+str(y_pred[k])+'\nConfidence: '+str(confidence[k])+'\nActual class: '+str(y_test[k]))
    plt.imshow(img_to_predict)

    for i in range(len(img_class_df)):
        if i < (col-1):
            img_class_path = img_class_df.iloc[i,2]
            img_class = img_read_resize(img_class_path)
            plt.subplot(1, col, i+2)
            plt.xticks([])
            plt.yticks([])
            plt.title('Image of class '+ str(y_pred[k]))
            plt.xlabel(str())
            plt.imshow(img_class)
        else:
            break;

    plt.show()

for k in range(len(X_test)):
    plot_prediction_examples(k)